In [ ]:
import subprocess, sys

def pip(args):
    subprocess.run([sys.executable, '-m', 'pip'] + args,
                   check=True, capture_output=True)

pip(['install', '-q', 'transformers>=4.51.0', '--upgrade'])
pip(['install', '-q', 'qwen-vl-utils', 'torchvision'])
pip(['install', '-q', 'packaging', 'datasets', 'huggingface_hub', 'hf_transfer'])
pip(['install', '-q', 'rouge_score', 'evaluate', 'underthesea', 'bert-score', 'nltk'])
pip(['uninstall', 'numpy', '-y'])
pip(['install', '-q', 'numpy==1.26.4'])

import transformers, numpy as np
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {np.__version__}')
print('\n Done — Restart kernel trước khi chạy Cell 2!')

In [ ]:
from pathlib import Path
import torch, os

KAGGLE_WORKING         = Path('/kaggle/working')
SRC_DATASET            = '/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset'
DST_DATASET            = str(KAGGLE_WORKING / 'vi_chart_dataset')
VIETNAMESE_DATA_PATH   = Path('/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese')
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / 'images'
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / 'viet_chart_vqa.jsonl'

MODEL_DIR     = KAGGLE_WORKING / 'qwen3vl_chartqa_model'
MODEL_REPO_ID = 'Nhaass/Qwen3-VL-2B-ChartQA'

CHART_TEST_N   = 500
VIETNAMESE_N   = 200
EVAL_TOTAL     = CHART_TEST_N + VIETNAMESE_N
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 64
METRICS        = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']

n_gpu = torch.cuda.device_count()
print(f'GPU count: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} — {p.total_memory // 1024**3} GB')
print(f'\nEval target : {EVAL_TOTAL} samples')
print(f'Model repo  : {MODEL_REPO_ID}')

In [ ]:
from huggingface_hub import snapshot_download
import shutil, os

if MODEL_DIR.exists() and not any(MODEL_DIR.iterdir()):
    shutil.rmtree(str(MODEL_DIR))
    print('Xóa folder rỗng')

if not MODEL_DIR.exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {MODEL_REPO_ID} ...')
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(MODEL_DIR),
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    print('Download xong')
else:
    print(f'Model đã có — {len(list(MODEL_DIR.iterdir()))} files')

print('Files:', os.listdir(str(MODEL_DIR))[:10])

In [ ]:
from datasets import load_from_disk
from PIL import Image
import json, shutil, os

def is_dataset_complete(dst, src):
    if not os.path.exists(dst):
        return False
    src_files = {os.path.relpath(os.path.join(r, f), src)
                 for r, _, fs in os.walk(src) for f in fs}
    dst_files = {os.path.relpath(os.path.join(r, f), dst)
                 for r, _, fs in os.walk(dst) for f in fs}
    missing = src_files - dst_files
    if missing:
        print(f'Thiếu {len(missing)} files')
        return False
    return True

if is_dataset_complete(DST_DATASET, SRC_DATASET):
    print('Dataset đã copy đầy đủ, skip.')
else:
    if os.path.exists(DST_DATASET):
        shutil.rmtree(DST_DATASET)
    print('Copying dataset ...')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy xong.')

vi_chart_dataset = load_from_disk(DST_DATASET)
print(vi_chart_dataset)

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))
print(f'Vietnamese records: {len(vietnamese_records)} loaded')

In [ ]:
from PIL import Image

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        return {'role': role, 'content': str(turn.get('content') or turn.get('value', ''))}
    return {'role': 'assistant', 'content': str(turn)}

chart_test_raw   = vi_chart_dataset['test']
chart_n          = min(CHART_TEST_N, len(chart_test_raw))
chart_test_items = [chart_test_raw[i] for i in range(chart_n)]
print(f'vi_chart test   : {chart_n} samples')

vn_test_items = []
for record in reversed(vietnamese_records):
    if len(vn_test_items) >= VIETNAMESE_N:
        break
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'Warning: {img_path}: {e}')
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs)-1, 2)]
    for idx, (q, a) in enumerate(pairs):
        if len(vn_test_items) >= VIETNAMESE_N:
            break
        rid = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_test_items.append({'id': rid, 'image': image, 'conversations': [q, a]})

print(f'vietnamese test  : {len(vn_test_items)} samples')
eval_dataset = chart_test_items + vn_test_items
print(f'Total            : {len(eval_dataset)} samples')

has_img = sum(1 for x in eval_dataset if x.get('image') is not None)
print(f'Samples có ảnh   : {has_img} / {len(eval_dataset)}')

In [ ]:
def evaluate_qwen3vl_chartqa(eval_dataset, model_dir, batch_size=1, max_new_tokens=64):
    import os, gc, torch, nltk
    import pandas as pd
    from transformers import AutoProcessor, AutoModelForImageTextToText
    from qwen_vl_utils import process_vision_info
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score as nltk_meteor
    from rouge_score import rouge_scorer
    from underthesea import word_tokenize
    from tqdm import tqdm
    from PIL import Image

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    nltk_path = '/usr/share/nltk_data'
    os.makedirs(nltk_path, exist_ok=True)
    nltk.data.path.append(nltk_path)
    for pkg in ['punkt', 'wordnet', 'omw-1.4']:
        nltk.download(pkg, download_dir=nltk_path, quiet=True)

    processor = AutoProcessor.from_pretrained(
        str(model_dir),
        trust_remote_code=True,
        min_pixels=128 * 28 * 28,
        max_pixels=256 * 28 * 28,
    )

    try:
        mdl = AutoModelForImageTextToText.from_pretrained(
            str(model_dir),
            torch_dtype=torch.bfloat16,
            device_map={'': device},
            trust_remote_code=True,
        ).eval()
        print('Loaded via AutoModelForImageTextToText')
    except Exception as e:
        print(f'AutoModelForImageTextToText failed ({e}), fallback Qwen2_5_VL...')
        from transformers import Qwen2_5_VLForConditionalGeneration
        mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            str(model_dir),
            torch_dtype=torch.bfloat16,
            device_map={'': device},
            trust_remote_code=True,
        ).eval()
        print('Loaded via Qwen2_5_VLForConditionalGeneration')

    items = [i for i in eval_dataset
             if all(k in i for k in ['id', 'conversations'])]
    print(f'  Model loaded — {len(items)} samples')

    system_msg = (
        'Bạn là chuyên gia phân tích biểu đồ. '
        'Hãy quan sát kỹ biểu đồ trong ảnh và trả lời câu hỏi bằng tiếng Việt '
        'một cách ngắn gọn, chính xác. Chỉ trả lời phần đáp án, không giải thích dài dòng.'
    )

    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1
    results  = []

    def get_pil_image(item):
        img = item.get('image')
        if img is None:
            return None
        if isinstance(img, Image.Image):
            return img
        elif isinstance(img, dict) and 'bytes' in img:
            import io
            return Image.open(io.BytesIO(img['bytes'])).convert('RGB')
        elif isinstance(img, (str, os.PathLike)):
            return Image.open(img).convert('RGB')
        return None

    def build_messages(it):
        question = str(it['conversations'][0]['content'])
        pil_img  = get_pil_image(it)
        content  = []
        if pil_img is not None:
            content.append({'type': 'image', 'image': pil_img})
        content.append({'type': 'text', 'text': f'Câu hỏi: {question}'})
        return [
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': content},
        ]

    def run_inference(messages):
        """Chạy inference với Qwen3-VL, tắt thinking mode."""
        try:
            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,   # Qwen3-VL thinking mode — tắt cho VQA
            )
        except TypeError:
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

        image_inputs, video_inputs = process_vision_info(messages)
        enc = processor(
            text=[text],
            images=image_inputs if image_inputs else None,
            videos=video_inputs if video_inputs else None,
            padding=True,
            return_tensors='pt',
        ).to(device)

        pad_id = (processor.tokenizer.pad_token_id
                  if processor.tokenizer.pad_token_id is not None
                  else processor.tokenizer.eos_token_id)

        with torch.no_grad():
            out = mdl.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=pad_id,
            )

        input_len = enc['input_ids'].shape[1]
        response  = processor.tokenizer.decode(
            out[0][input_len:], skip_special_tokens=True
        ).strip()

        import re
        response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

        return response, enc

    for it in tqdm(items, desc='Evaluating Qwen3-VL-ChartQA'):
        try:
            messages = build_messages(it)
            response, enc = run_inference(messages)

        except torch.cuda.OutOfMemoryError:
            print(f'  OOM sample {it["id"]} — fallback text-only')
            torch.cuda.empty_cache()
            gc.collect()
            question = str(it['conversations'][0]['content'])
            messages_text = [
                {'role': 'system', 'content': system_msg},
                {'role': 'user',   'content': f'Câu hỏi: {question}'},
            ]
            try:
                text = processor.apply_chat_template(
                    messages_text,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                text = processor.apply_chat_template(
                    messages_text, tokenize=False, add_generation_prompt=True
                )
            enc = processor(text=[text], return_tensors='pt').to(device)
            with torch.no_grad():
                out = mdl.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id,
                )
            input_len = enc['input_ids'].shape[1]
            import re
            response = processor.tokenizer.decode(
                out[0][input_len:], skip_special_tokens=True
            ).strip()
            response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

        finally:
            torch.cuda.empty_cache()
            gc.collect()

        gt  = str(it['conversations'][1]['content'])
        ref = word_tokenize(gt, format='text').split()
        hyp = word_tokenize(response, format='text').split() if response else ['']

        bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        meteor = float(nltk_meteor([ref], hyp))
        r      = scorer.score(gt, response)

        results.append({
            'id':           it['id'],
            'question':     str(it['conversations'][0]['content']),
            'ground_truth': gt,
            'response':     response,
            'bleu':         bleu,
            'meteor':       meteor,
            'rouge1':       r['rouge1'].fmeasure,
            'rouge2':       r['rouge2'].fmeasure,
            'rougeL':       r['rougeL'].fmeasure,
        })

    df = pd.DataFrame(results)

    print('\nComputing BERTScore ...')
    try:
        import bert_score as bs_lib
        _, _, F1 = bs_lib.score(
            df['response'].tolist(),
            df['ground_truth'].tolist(),
            lang='vi', verbose=False, rescale_with_baseline=False,
        )
        df['bertscore'] = F1.tolist()
    except Exception as e:
        print(f'BERTScore failed: {e}')
        df['bertscore'] = [0.0] * len(df)

    metrics = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']
    avg = {m: df[m].mean() for m in metrics}
    return df, avg

print('Worker function defined ')

In [ ]:
print('=' * 60)
print(f'Evaluating Qwen3-VL-2B-ChartQA  [{len(eval_dataset)} samples]')
print(f'batch_size={BATCH_SIZE}  max_new_tokens={MAX_NEW_TOKENS}')
print('=' * 60)

df_qwen3, avg_qwen3 = evaluate_qwen3vl_chartqa(
    eval_dataset,
    model_dir=MODEL_DIR,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)

print('\nKết quả Qwen3-VL-2B-ChartQA:')
for k, v in avg_qwen3.items():
    print(f'  {k:<12}: {v:.4f}')

csv_out = KAGGLE_WORKING / 'debug_qwen3vl_chartqa.csv'
df_qwen3.to_csv(str(csv_out), index=False, encoding='utf-8')
print(f'\nSaved: {csv_out}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

all_results = {
    'Qwen3-VL-2B-ChartQA': (df_qwen3, avg_qwen3),
}

rows = []
for mname, (_, avg) in all_results.items():
    row = {'Model': mname}
    for m in METRICS:
        row[m.upper()] = round(avg.get(m, 0.0), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')

def highlight_best(s):
    return ['background-color: #d4edda; font-weight: bold'
            if v == s.max() else '' for v in s]

def highlight_worst(s):
    return ['background-color: #f8d7da'
            if v == s.min() else '' for v in s]

print(f'KẾT QUẢ ({EVAL_TOTAL} samples | Xanh = cao nhất | Đỏ = thấp nhất)\n')
display(summary_df.style.apply(highlight_best).apply(highlight_worst))

csv_summary = KAGGLE_WORKING / 'eval_qwen3vl_chartqa_summary.csv'
summary_df.reset_index().to_csv(str(csv_summary), index=False, encoding='utf-8')
print(f'Saved: {csv_summary}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n      = len(all_results)
x      = np.arange(len(METRICS))
width  = 0.8 / max(n, 1)
colors = ['#2980b9', '#27ae60', '#e67e22', '#8e44ad', '#d65f5f']

fig, ax = plt.subplots(figsize=(13, 5))
for idx, (mname, (_, avg)) in enumerate(all_results.items()):
    offset = idx * width - (n - 1) * width / 2
    vals   = [avg.get(m, 0.0) for m in METRICS]
    bars   = ax.bar(x + offset, vals, width, label=mname, color=colors[idx % len(colors)])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in METRICS], fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.2)
ax.set_title(f'Qwen3-VL-2B-ChartQA — Eval Results  ({EVAL_TOTAL} samples)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()

img_path = KAGGLE_WORKING / 'eval_qwen3vl_chartqa_bar.png'
plt.savefig(str(img_path), dpi=150)
plt.show()
print(f'Saved chart: {img_path}')

In [ ]:
import pandas as pd

df_analysis = df_qwen3[['id', 'question', 'ground_truth', 'response'] + METRICS].copy()

print(f'Tổng samples: {len(df_analysis)}')
print(f'\nAverage scores:')
for m in METRICS:
    if m in df_analysis.columns:
        print(f'  {m:<12}: {df_analysis[m].mean():.4f}')

print('\n--- Top 10 sample model trả lời TỐT NHẤT (BERTScore cao) ---')
display(df_analysis.nlargest(10, 'bertscore')
        [['id', 'question', 'ground_truth', 'response', 'bertscore', 'rouge1', 'bleu']])

print('\n--- Top 10 sample model trả lời KÉM NHẤT (BERTScore thấp) ---')
display(df_analysis.nsmallest(10, 'bertscore')
        [['id', 'question', 'ground_truth', 'response', 'bertscore', 'rouge1', 'bleu']])

analysis_csv = KAGGLE_WORKING / 'debug_qwen3vl_chartqa_analysis.csv'
df_analysis.to_csv(str(analysis_csv), index=False, encoding='utf-8')
print(f'\nSaved: {analysis_csv}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, m in enumerate(METRICS):
    ax = axes[i]
    if m in df_qwen3.columns:
        ax.hist(df_qwen3[m], bins=30, alpha=0.8,
                color='#2980b9', label='Qwen3-VL-2B-ChartQA')
    ax.set_title(m.upper(), fontsize=11)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle(
    f'Phân phối score: Qwen3-VL-2B-ChartQA  ({len(df_qwen3)} samples)',
    fontsize=13,
)
plt.tight_layout()

hist_path = KAGGLE_WORKING / 'histogram_qwen3vl_chartqa.png'
plt.savefig(str(hist_path), dpi=150)
plt.show()
print(f'Saved: {hist_path}')